In [ ]:
# In[1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import joblib


# In[3]:


df = pd.read_csv("fraudTest.csv")

print("Dataset Shape:", df.shape)
df.head()


# In[4]:


drop_cols = ['Unnamed: 0', 'trans_date_trans_time', 'cc_num']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')


# In[5]:


df = df.dropna()


# In[6]:


df.isnull().sum()


# In[7]:


label_encoders = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le


# In[8]:


X = df.drop('is_fraud', axis=1)
y = df['is_fraud']


# In[9]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# In[11]:


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# In[12]:


lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)


# In[13]:


dt_model = DecisionTreeClassifier(max_depth=10)
dt_model.fit(X_train, y_train)


# In[14]:


rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


# In[15]:


def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    
    print(f"\n{name} Results")
    print("-" * 40)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

evaluate_model(lr_model, X_test, y_test, "Logistic Regression")
evaluate_model(dt_model, X_test, y_test, "Decision Tree")
evaluate_model(rf_model, X_test, y_test, "Random Forest")


# In[16]:


joblib.dump(rf_model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved successfully!")

